Phase 6: Decryption and Verification at Receiver

System Imports

In [14]:
import oqs
import time
import os
import matplotlib.pyplot as plt
import numpy as np
import random
import threading
import concurrent.futures
import queue
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
import struct
from cryptography.hazmat.primitives.ciphers.aead import ChaCha20Poly1305
import hashlib
from cryptography.hazmat.primitives.ciphers.aead import AESGCM


Demo Parameters

In [15]:
kyber_alg = "Kyber768"
dilithium_alg = "Dilithium3"
n_records = 500  # High-traffic scenario
num_partitions = 3
delta = 2
num_nodes = 2
enclaves_per_node = 2
device_secret = {
    "Device_1": os.urandom(32)
}
policy = {
    "role": ["doctor", "researcher"],
    "region": ["EU", "US"]
}

# Performance monitoring dictionary
performance_metrics = {
    "phase_2": {},
    "phase_3": {},
    "phase_4": {},
    "phase_6": {}
}

Phase 1

In [16]:
# HDFS generate key pair
kem = oqs.KeyEncapsulation(kyber_alg)
hdfs_PKs = kem.generate_keypair()
hdfs_SKs = kem.export_secret_key()
print("HDFS PK:", hdfs_PKs)
print("HDFS SK:", hdfs_SKs)

#key encapsulation
hdfs_ciphertext, hdfs_shared_secret_enc = kem.encap_secret(hdfs_PKs)
print("HDFS Ciphertext:", hdfs_ciphertext)
print("HDFS Shared Secret (Encapsulated):", hdfs_shared_secret_enc)


HDFS PK: b'ZS\x14\xb6i^\x00\x11z\xf7\x94|\x83*&\x1d40%[\x9e\xde\xd8\xbb\xe2A`_\x89bH\xa4xR\x9b\xca\xf7ah\xc8*\x97\x87f\x06r\x95X`\xb5\x7f\\,\xc2\x9c\xdc\xc8\xb0\xecZme\xc3\xc8,k\xec\xd4"\x8f|\x99\x92\x91\'\xaeWL\x84\xd4\x99\x9at\x0f\xcf\xa0E\xcaqf>\xd1\xba\xc0\xf9\xa3\xbc\x85\x07F\x02\xbf\xfe\xa70X3|\x11p\x89\xdf\xecD\x14\xfao\xd3t\x98\xdb\xb0`s\x17\x0f\x8a\x03\xa1t\x9bZ\xe7\x03]\t\xd1\n\x96\xc2\x9d\xd4\xbb\x01h\xa6}\xb1@\x8c\x80c\x07"\xb7 \t*\xcd\xfc\xf7\xb0+\x1a\x80\x8a\xc1\xa5\xd5\xd4|#\x83\xa1\xealm\xa3\x8cU^\x08\x8c\xec\x98P\x8bKd^i\xb6\xd0\xe1\xa9\xe5\xb7X\xb2\xea[\xc9\x83\x8f\xc4\xc3P9\x17K\xd3vo"EU\x8d\xb7\xbe\xaet\xc3Kcz\xaf\xb8\xbap\xb5\x8d8W\xca5z\xc2\xa7g\xb8jC\x90\x9a\xd5\x8a\xd5\x81mtwh\x11H?\x0b\x03(\xfck\xa8\x07\x07\x10\x848\x8b\xc2\x14g\r\xc2\xa6\xbd\x88\xa2\xb4\x05\x05]\xb3\xab\x0f\xb2d\xd5\xf1=Z\xd1\x01\xc0\xd2\xc9\xc3)\xbcn\xb1\xb5\xf1\xba\xaf\xdf\x987\xd9\xb2j\x1c\x80$\xe0T\t\xder4Ks\xa2m\xa9H\x10\xd5}Y\xd8@"\xa8\xbfq\xe6t\xbb\x08\x9d\xa1\\\x8aQZ\xc7c\x9a\x0c\n\xaa

Phase 2

In [17]:
#Data acquisition
# Simulate data acquisition by generating random data

class IoTDevice:
    def __init__(self, device_id, SS):
        self.device_id = device_id
        self.counter = 0
        self.SS = SS
        self.ds = device_secret.get(device_id)  # Simulate device-specific secret for PUF
        self.ce = "epoch1"  # Simulate current epoch for authentication
        self.last_message = None  # Store the last message for potential retries

    def generate_payload(self, topic):
        if topic == "temperature":
            return round(random.uniform(20.0, 35.0), 2)
        elif topic == "humidity":
            return round(random.uniform(30.0, 80.0), 2)
        elif topic == "heart_rate":
            return random.randint(60, 120)
        else:
            return random.randint(0, 100)
    
    def encrypt_record(self, Ki, nounce, mi, aad):
        aead = ChaCha20Poly1305(Ki)
        plaintext = str(mi).encode()
        ciphertext = aead.encrypt(nounce, plaintext, aad)
        return ciphertext
    
    def simulate_puf(self, device_secret, ce):
        data = device_secret + ce.encode()
        return hashlib.sha256(data).digest()
    
    def hash_ciphertext(self, Ci):
        return hashlib.sha256(Ci).digest()

    def generate_auth(self, Ri, ce, ri_e, Ci):
        data = (
            Ri['ID'].encode() + 
            struct.pack(">d", Ri['ts']) + 
            Ri['ctr'].to_bytes(8, 'big') + 
            ce.encode() + 
            ri_e + 
            self.hash_ciphertext(Ci)
        )
        return hashlib.sha256(data).digest()
    
    def sense(self):
        if self.last_message and random.random() < 0.3:  # Simulate 30% chance of retrying the last message
            print("Retrying last message due to simulated sensing error...")
            return self.last_message
        topics = ["temperature", "humidity", "heart_rate"]
        topic = random.choice(topics)
        ts = time.time()
        Ri = {
            "ID": self.device_id,
            "ts": ts,
            "ctr": self.counter,
            "topic": topic,
            "m": self.generate_payload(topic)
        }
    
        ts_bytes = struct.pack(">d", ts)          # float -> 8-byte IEEE 754
        device_id_bytes = self.device_id.encode() # str -> bytes

        Ki = HKDF(
            algorithm=hashes.SHA256(),
            length=32,
            salt=None,
            info=b'IoT data encryption',
        ).derive(device_id_bytes + ts_bytes + self.counter.to_bytes(8, 'big') + self.SS)

        # Generate nounce 
        nounce = os.urandom(12)

        self.counter += 1

        #Construct AEAD record (Ri, Ki, nounce)
        add_str = f"{Ri['ID']}|{Ri['ts']}|{Ri['ctr']}|{Ri['topic']}|{Ri['m']}"
        aad = add_str.encode()

        #Build protected record
        Ci = self.encrypt_record(Ki, nounce, Ri['m'], aad)
               
        #Generate authentication tag
        ri_e = self.simulate_puf(self.ds, ce=self.ce)
        Auth = self.generate_auth(Ri, ce=self.ce, ri_e=ri_e, Ci=Ci)

        final_message = {
            "ID": Ri['ID'],
            "ts": Ri['ts'],
            "ctr": Ri['ctr'],
            "topic": Ri['topic'],
            "nounce": nounce,
            "AAD": aad,
            "Ci": Ci,
            "ce": self.ce,
            "Auth": Auth
            }
        
        time.sleep(random.uniform(0, 1))  # Simulate sensing delay

        self.last_message = final_message  # Store the last message for potential retries
        return final_message
    
# Simulate 1 IoT device
device = IoTDevice("Device_1", hdfs_shared_secret_enc)


Phase 3

In [18]:
# Simulate Kafka Stream Ingestion

class KafkaSimulator:
    def __init__(self, num_partitions=3):
        self.partitions = {i: [] for i in range(num_partitions)}
        self.num_partitions = num_partitions
        self.lock = threading.Lock()
    
    def partition_fn(self, ID, topic):
        # Simple hash-based partitioning
        return hash(ID + topic) % self.num_partitions
    
    def publish(self, Mi):
        p = self.partition_fn(Mi['ID'], Mi['topic'])
        with self.lock:
            self.partitions[p].append(Mi)
        print(f"Published message to partition {p}: {Mi}")

    def get_partition(self):
        with self.lock:
            return {p: list(msgs) for p, msgs in self.partitions.items()}

# Simulate Spark Stream Processing

class SparkSimulator:
    def __init__(self, delta):
        self.delta = delta
        self.epoch_batches = {}
        self.epoch_seen = {}
    
    def extract_metadata(self, Mi):
        metadata = {
            "ID": Mi['ID'],
            "ts": Mi['ts'],
            "ctr": Mi['ctr'],
            "topic": Mi['topic'],
            "ce": Mi['ce']
        }
        return metadata
    
    def process_stream(self, kafka_partitions):
        metadata_stream = {}
        for partition, messages in kafka_partitions.items():
            metadata_stream[partition] = []
            for Mi in messages:
                metadata = self.extract_metadata(Mi)
                metadata_stream[partition].append(metadata)
                print(f"Extracted metadata from partition {partition}: {metadata}")
        return metadata_stream
    
    def compute_epoch(self, ts):
        return int(ts // self.delta)
    
    def assign_epoch(self, Mi):
        e = self.compute_epoch(Mi['ts'])
        return e
    
    def form_batches(self, kafka_partitions):
        batches = {}
        for partition, messages in kafka_partitions.items():
            for Mi in messages:
                e = self.assign_epoch(Mi)
                batches.setdefault(e, []).append(Mi)

        for e, msgs in batches.items():
            batches[e] = self.deduplicate(msgs)

        return batches
    
    def compute_rid(self, Mi):
        data = (
            Mi['ID'].encode() +
            struct.pack(">d", Mi['ts']) + 
            Mi['ctr'].to_bytes(8, 'big') + 
            Mi['topic'].encode()
        )
        return hashlib.sha256(data).hexdigest()
    
    def deduplicate(self, batch):
        seen_rids = set()
        unique_batch = []
        for Mi in batch:
            rid = self.compute_rid(Mi)
            if rid not in seen_rids:
                seen_rids.add(rid)
                unique_batch.append(Mi)
            else:
                print(f"Duplicate detected and removed: {Mi}")
        return unique_batch
    
    def derive_context(self, batch):
        topics = set(msg["topic"] for msg in batch)
        if "heart_rate" in topics:
            return "healthcare"
        elif "temperature" in topics and "humidity" in topics:
            return "environment"
        else:
            return "generic"
    
    def map_policy(self, ctx):
        policy_map = {
            "healthcare": "POLICY_HEALTH_STRICT",
            "environment": "POLICY_ENV_MONITOR",
            "generic": "POLICY_DEFAULT"
        }
        return policy_map.get(ctx, "POLICY_DEFAULT")
    
    def assign_policy(self, epoch_batches):
        policy_results = {}
        for e, batch in epoch_batches.items():
            ctx = self.derive_context(batch)
            policy = self.map_policy(ctx)
            policy_results[e] = {
                "context": ctx,
                "policy": policy,
                "batch": batch
            }
        return policy_results
    
    def hash_token(self, value):
        return hashlib.sha256(value.encode()).hexdigest()
    
    def generate_token(self, Mi, epoch, context):
        tokens = {
            "kw1": self.hash_token(Mi["ID"]),
            "kw2": self.hash_token(Mi["topic"]),
            "kw3": self.hash_token(str(epoch)),
            "kw4": self.hash_token(context + "|regionA")
        }
        return tokens
    
    def tokenize_batches(self, policy_batches):
        tokenized = {}
        for e, data in policy_batches.items():
            ctx = data["context"]
            batch = data["batch"]
            tokenized[e] = []
            for Mi in batch:
                tokens = self.generate_token(Mi, e, ctx)
                tokenized[e].append({
                    "Mi": Mi,
                    "tokens": tokens
                })
        return tokenized

    def compute_obj_id(self, Mi):
        hC = hashlib.sha256(Mi["Ci"]).digest()
        data = (
            Mi["ID"].encode() +
            struct.pack(">d", Mi["ts"]) +
            str(Mi["ctr"]).encode() +
            hC
        )
        return hashlib.sha256(data).hexdigest()
    
    def build_index_entry(self, tokens, obj_id):
        return {
            "tokens": tokens,
            "obj_id": obj_id
        }
    
    def build_epoch_index(self, tokenized_batches):
        epoch_indices = {}
        for e, records in tokenized_batches.items():
            epoch_indices[e] = []
            for r in records:
                Mi = r["Mi"]
                tokens = r["tokens"]
                obj_id = self.compute_obj_id(Mi)
                idx_entry = self.build_index_entry(tokens, obj_id)
                epoch_indices[e].append(idx_entry)
        return epoch_indices
    
    def compute_pre_digest(self, Mi, epoch):
        data = (
            Mi["ID"].encode() +
            struct.pack(">d", Mi["ts"]) +
            str(Mi["ctr"]).encode() +
            str(epoch).encode() +
            Mi["Ci"]
        )
        return hashlib.sha256(data).hexdigest()

    def compute_epoch_digests(self, epoch_batches):
        epoch_digests = {}
        for e, batch in epoch_batches.items():
            digests = []
            for Mi in batch:
                h_i = self.compute_pre_digest(Mi, e)
                digests.append(h_i)
            epoch_digests[e] = digests
        return epoch_digests
    
    def build_final_output(self, epoch_batches, epoch_indices, policy_batches, epoch_digests):
        final_output = {}
        for e in epoch_batches:
            De = epoch_batches[e]
            Ie = epoch_indices[e]
            H_e = epoch_digests[e]
            PolRef = policy_batches[e]["policy"]
            final_output[e] = {
                "De": De,
                "Ie": Ie,
                "H_e": H_e,
                "PolRef": PolRef,
                "epoch": e
            }
        return final_output

class Multiprocessor:
    def __init__(self, kafka_simulator, spark_simulator, kafka_workers=4, spark_workers=3):
        self.kafka_simulator = kafka_simulator
        self.spark_simulator = spark_simulator
        self.kafka_workers = kafka_workers
        self.spark_workers = spark_workers

    def publish_messages(self, messages):
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.kafka_workers) as executor:
            futures = [executor.submit(self.kafka_simulator.publish, message) for message in messages]
            for future in concurrent.futures.as_completed(futures):
                future.result()

    def process_partitions(self):
        partitions = self.kafka_simulator.get_partition()
        metadata_stream = {}
        epoch_batches = {}

        with concurrent.futures.ThreadPoolExecutor(max_workers=self.spark_workers) as executor:
            futures = {
                executor.submit(self._process_partition, partition, messages): partition
                for partition, messages in partitions.items()
            }
            for future in concurrent.futures.as_completed(futures):
                partition = futures[future]
                metadata, partition_batches = future.result()
                metadata_stream[partition] = metadata
                for e, msgs in partition_batches.items():
                    epoch_batches.setdefault(e, []).extend(msgs)

        for e, msgs in list(epoch_batches.items()):
            epoch_batches[e] = self.spark_simulator.deduplicate(msgs)

        return metadata_stream, epoch_batches

    def _process_partition(self, partition, messages):
        metadata = []
        batches = {}
        for Mi in messages:
            metadata.append(self.spark_simulator.extract_metadata(Mi))
            e = self.spark_simulator.assign_epoch(Mi)
            batches.setdefault(e, []).append(Mi)
        return metadata, batches

    def build_final_output(self, epoch_batches):
        policy_results = self.spark_simulator.assign_policy(epoch_batches)
        tokenized_batches = self.spark_simulator.tokenize_batches(policy_results)
        epoch_indices = self.spark_simulator.build_epoch_index(tokenized_batches)
        epoch_digests = self.spark_simulator.compute_epoch_digests(epoch_batches)
        final_output = self.spark_simulator.build_final_output(
            epoch_batches, epoch_indices, policy_results, epoch_digests
        )
        return {
            "policy_results": policy_results,
            "tokenized_batches": tokenized_batches,
            "epoch_indices": epoch_indices,
            "epoch_digests": epoch_digests,
            "final_output": final_output
        }

# Outputs

# Phase 3 Performance Monitoring

t_phase3_start = time.time()
t_kafka_start = time.time()

kafka_simulator = KafkaSimulator(num_partitions)
spark_simulator = SparkSimulator(delta)
multiprocessor = Multiprocessor(
    kafka_simulator,
    spark_simulator,
    kafka_workers=6,
    spark_workers=3
)

iot_messages = []
for i in range(n_records):
    iot_messages.append(device.sense())

multiprocessor.publish_messages(iot_messages)

t_kafka_end = time.time()
t_spark_start = time.time()

print("Simulating Spark stream processing in parallel...")
metadata_stream, batches = multiprocessor.process_partitions()

t_spark_batch_end = time.time()
t_spark_policy_start = time.time()

final_components = multiprocessor.build_final_output(batches)
policy_results = final_components["policy_results"]
tokenized_batches = final_components["tokenized_batches"]
epoch_indices = final_components["epoch_indices"]
epoch_digests = final_components["epoch_digests"]
final_output = final_components["final_output"]


t_spark_policy_end = time.time()
t_spark_output_start = time.time()

t_spark_output_end = time.time()
t_phase3_end = time.time()

# Store timing metrics
performance_metrics["phase_3"] = {
    "kafka_ingestion": t_kafka_end - t_kafka_start,
    "spark_batch_formation": t_spark_batch_end - t_spark_start,
    "spark_policy_tokenization": t_spark_policy_end - t_spark_policy_start,
    "spark_output_generation": t_spark_output_end - t_spark_output_start,
    "total_phase3": t_phase3_end - t_phase3_start,
    "total_messages": n_records,
    "num_epochs": len(final_output)
}


Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing error...
Retrying last message due to simulated sensing e

Phase 4

In [19]:
# Simulate Enclave

class Enclave:
    def __init__(self, enclave_id):
        self.id = enclave_id
        self._secure_memory = None
        self.singer = oqs.Signature(dilithium_alg)
        self.public_key = self.singer.generate_keypair()
    
    def load_batch(self, Pi_e):
        print(f"[Enclave {self.id}] Loading batch into secure memory for epoch {Pi_e['epoch']}")
        self._secure_memory = Pi_e.copy()

    def execute(self):
        if self._secure_memory is None:
            raise Exception("No batch loaded in enclave")
        print(f"[Enclave {self.id}] Executing inside TEE")
        Pi_e = self._secure_memory
        return self._process_inside_enclave(Pi_e)
    
    def _process_inside_enclave(self, Pi_e):
        print(f"[Enclave {self.id}] PUF verification")
        V_e = self.verify_batch(Pi_e)
        print(V_e)
        print(f"[Enclave {self.id}] Epoch key generation")
        Ke = self.generate_epoch_key()
        print(Ke)
        ctxe = self.compute_ctx(
            Pi_e["epoch"],
            Pi_e["PolRef"],
            Pi_e["H_e"]
        )
        print(ctxe)
        print(f"[Enclave {self.id}] Batch AEAD protection")
        Cagg_e = self.aggregate_ciphertexts(V_e)
        IVe, CTe = self.encrypt_batch(Ke, Cagg_e, ctxe)
        print(CTe)
        print(f"[Enclave {self.id}] Kyber KEM Protection")
        Kp = self.derive_key(hdfs_shared_secret_enc)
        EncKe = self.protect_key(Ke, Kp)
        print(EncKe)
        print(f"[Enclave {self.id}] CP-ABE Encryption")
        CT_ABE = self.cpabe_encrypt(Pi_e["PolRef"], EncKe)
        print(CT_ABE)
        print(f"[Enclave {self.id}] Batch commitment")
        h_e = self.compute_final_digest(Pi_e["epoch"], IVe, CTe, CT_ABE, hdfs_ciphertext)
        print(h_e)
        print(f"[Enclave {self.id}] Dilithium signing")
        sigma_e = self.sign_batch(h_e)
        print(sigma_e)
        return {
            "epoch": Pi_e["epoch"],
            "valid_records": V_e,
            "Ke": Ke,
            "ctxe": ctxe,
            "IVe": IVe,
            "CTe": CTe,
            "num_records": len(V_e),
            "CT_Ke": hdfs_ciphertext,
            "CT_ABE": CT_ABE,
            "h_e": h_e,
            "root": self.merkle_root(Pi_e["H_e"]),
            "index": Pi_e.get("Ie", None),
            "sigma_e": sigma_e
        }
    
    def verify_batch(self, Pi_e):
        valid_records = []
        for Mi in Pi_e["De"]:
            r_ie = self.verify_puf(Mi["ID"], Mi["ce"])
            if r_ie is None:
                continue
            expected_auth = self._compute_auth(Mi, r_ie)
            if expected_auth == Mi["Auth"]:
                valid_records.append(Mi)
            else:
                print(f"[Enclave {self.id}] Auth failed for ctr={Mi['ctr']}")
        return valid_records

    def _compute_auth(self, Mi, r_ie):
        hC = hashlib.sha256(Mi["Ci"]).digest()
        data = (
            Mi["ID"].encode() +
            struct.pack(">d", Mi["ts"]) +
            Mi["ctr"].to_bytes(8, 'big') +
            Mi["ce"].encode() + r_ie + hC
        )
        return hashlib.sha256(data).digest()
    
    def verify_puf(self, device_id, ce):
        secret = device_secret.get(device_id)
        if secret is None:
            return None
        data = secret + ce.encode()
        return hashlib.sha256(data).digest()

    def generate_epoch_key(self):
        return os.urandom(32)
    
    def compute_digest_root(self, H_e):
        combined = b''.join(h.encode() for h in H_e)
        return hashlib.sha256(combined).digest()
    
    def compute_ctx(self, epoch, policy, H_e):
        digest_root = self.compute_digest_root(H_e)
        data = (
            b"CTX" +
            str(epoch).encode() +
            policy.encode() +
            digest_root
        )
        return hashlib.sha256(data).hexdigest()

    def aggregate_ciphertexts(self, valid_records):
        agg = b''
        for Mi in valid_records:
            Ci = Mi["Ci"]
            length = len(Ci).to_bytes(4, 'big')
            agg += length + Ci
        return agg
    
    def encrypt_batch(self, Ke, Cagg_e, ctxe):
        aesgcm = AESGCM(Ke)
        IVe = os.urandom(12)
        CTe = aesgcm.encrypt(IVe, Cagg_e, ctxe.encode())
        return IVe, CTe
    
    def derive_key(self, shared_secret):
        return hashlib.sha256(shared_secret).digest()
    
    def protect_key(self, Ke, Kp):
        return bytes(a ^ b for a, b in zip(Ke, Kp))

    def cpabe_encrypt(self, policy, EncKe):
        return {
            "policy": policy,
            "ciphertext": EncKe
        }
    
    def compute_final_digest(self, epoch, IVe, CTe, CT_ABE, CT_Ke):
        data = (
            str(epoch).encode() +
            IVe +
            CTe +
            CT_ABE["ciphertext"] +
            CT_ABE["policy"].encode() +
            CT_Ke
        )
        return hashlib.sha256(data).hexdigest()
    
    def merkle_root(self, hashes):
        nodes = [h.encode() for h in hashes]
        while len(nodes) > 1:
            temp = []
            for i in range(0, len(nodes), 2):
                left = nodes[i]
                right = nodes[i+1] if i+1 < len(nodes) else left
                temp.append(hashlib.sha256(left + right).digest())
            nodes = temp
        return nodes[0].hex()
    
    def sign_batch(self, h_e):
        signature = self.singer.sign(h_e.encode())
        return signature

class EdgeNode:
    def __init__(self, node_id, num_enclaves):
        self.node_id = node_id
        self.enclaves = [
            Enclave(f"{node_id}-{i}") for i in range(num_enclaves)
        ]
        self.load = 0

class EdgeCluster:
    def __init__(self, num_nodes, enclaves_per_node):
        self.nodes = [
            EdgeNode(f"Node_{i}", enclaves_per_node) for i in range(num_nodes)
        ]
        self.lock = threading.Lock()
    
    def route(self, Pi_e):
        with self.lock:
            node = min(self.nodes, key=lambda n: n.load)
            enclave = random.choice(node.enclaves)
            node.load += 1
            print(f"[Routing] Epoch {Pi_e['epoch']} -> {node.node_id} -> Enclave {enclave.id}")
            return enclave

class EdgeWorkerPool:
    def __init__(self, cluster, edge_workers=4):
        self.cluster = cluster
        self.edge_workers = edge_workers

    def execute_epochs(self, final_output):
        edge_results = {}
        enclave_times = []

        with concurrent.futures.ThreadPoolExecutor(max_workers=self.edge_workers) as executor:
            futures = {
                executor.submit(self._process_epoch, epoch, Pi_e): epoch
                for epoch, Pi_e in final_output.items()
            }
            for future in concurrent.futures.as_completed(futures):
                epoch = futures[future]
                result = future.result()
                edge_results[epoch] = result
                enclave_times.append(result["elapsed"])

        return edge_results, enclave_times

    def _process_epoch(self, epoch, Pi_e):
        t_enclave_start = time.time()
        enclave = self.cluster.route(Pi_e)
        enclave.load_batch(Pi_e)
        result = enclave.execute()
        t_enclave_end = time.time()
        result["enclave_id"] = enclave.id
        result["elapsed"] = t_enclave_end - t_enclave_start
        return result

# Outputs

cluster = EdgeCluster(num_nodes, enclaves_per_node)
edge_pool = EdgeWorkerPool(cluster, edge_workers=len(cluster.nodes) * enclaves_per_node)

t_phase4_start = time.time()
edge_results, enclave_times = edge_pool.execute_epochs(final_output)
t_phase4_end = time.time()

# Keep at least one batch for Phase 6
result = next(iter(edge_results.values())) if edge_results else None

# Store Phase 4 metrics
performance_metrics["phase_4"] = {
    "total_phase4": t_phase4_end - t_phase4_start,
    "num_epochs": len(edge_results),
    "avg_epoch_time": sum(enclave_times) / len(enclave_times) if enclave_times else 0,
    "min_epoch_time": min(enclave_times) if enclave_times else 0,
    "max_epoch_time": max(enclave_times) if enclave_times else 0,
    "enclave_count": len(cluster.nodes) * enclaves_per_node
}


[Routing] Epoch 889047110 -> Node_0 -> Enclave Node_0-0
[Enclave Node_0-0] Loading batch into secure memory for epoch 889047110
[Enclave Node_0-0] Executing inside TEE
[Enclave Node_0-0] PUF verification
[{'ID': 'Device_1', 'ts': 1778094221.3655097, 'ctr': 3, 'topic': 'temperature', 'nounce': b'\xab6\xa5U\x17/LX\xf0\xb4\xeeI', 'AAD': b'Device_1|1778094221.3655097|3|temperature|22.98', 'Ci': b'k\xeei~o]T\xf0\xc0\x93~\xc0\xb3r\xaa\xbb\x1a\x17\x87\xe5T', 'ce': 'epoch1', 'Auth': b'\x97\xef\x05SnI\xa5\xbc6\x0b\xa5 \xed\xc2\xa9\xa2\x0b\x8c\xd7\xbcXj\x01\x08\xb97\xf9\x10\xa2\xb4\xef?'}, {'ID': 'Device_1', 'ts': 1778094221.9436383, 'ctr': 5, 'topic': 'temperature', 'nounce': b'qw\xbd\x1b\xb39~\x8c<K}I', 'AAD': b'Device_1|1778094221.9436383|5|temperature|20.61', 'Ci': b'\x92\x01\xf6\xf9\xcazb%7<\t\x1eN\x02\x0c\xa3\x1b0S\xc7*', 'ce': 'epoch1', 'Auth': b'\xfb\xa1\x14\x91"\xc0\x17#\xaftH\xe3L]\xf4k4\xa5R)u\xb8Tym\xe4\xf5He\xb4Zo'}, {'ID': 'Device_1', 'ts': 1778094221.1322157, 'ctr': 0, 'topic': 'h

Phase 5

In [20]:
# LocalFS, HDFS Simulation
def get_storage_path(base_dir, group_k, node_j, epoch):
    path = os.path.join(
        base_dir,
        "iot_stream",
        f"group_{group_k}",
        f"node_{node_j}",
        f"epoch_{epoch}"
    )

    os.makedirs(path, exist_ok=True)
    return path

def store_batch(self, Omega_e, base_dir, group_k, node_j):
    path = get_storage_path(base_dir, group_k, node_j, Omega_e["epoch"])

    # store each component
    with open(os.path.join(path, "CTe.bin"), "wb") as f:
        f.write(Omega_e["CTe"])

    with open(os.path.join(path, "IVe.bin"), "wb") as f:
        f.write(Omega_e["IVe"])

    with open(os.path.join(path, "CT_Ke.bin"), "wb") as f:
        f.write(Omega_e["CT_Ke"])

    with open(os.path.join(path, "sigma.bin"), "wb") as f:
        f.write(Omega_e["sigma_e"])

    # CP-ABE (structured)
    import json
    with open(os.path.join(path, "CT_ABE.json"), "w") as f:
        json.dump(Omega_e["CT_ABE"], f)

    return path

def store_index(self, Omega_e, base_dir, group_k, node_j):
    path = get_storage_path(base_dir, group_k, node_j, Omega_e["epoch"])

    import json
    with open(os.path.join(path, "index.json"), "w") as f:
        json.dump(Omega_e["index"], f)

def create_manifest(self, Omega_e, node_j, group_k):
    return {
        "epoch": Omega_e["epoch"],
        "node": node_j,
        "group": group_k,

        "CT_ABE": Omega_e["CT_ABE"],
        "CT_Ke": Omega_e["CT_Ke"],

        "sigma": Omega_e["sigma_e"],
        "h_e": Omega_e["h_e"]
    }

def store_manifest(self, manifest, base_dir, group_k, node_j):
    import json

    path = get_storage_path(base_dir, group_k, node_j, manifest["epoch"])

    with open(os.path.join(path, "manifest.json"), "w") as f:
        json.dump(manifest, f)

def build_index(self, De, epoch):
    index = []

    for Mi in De:
        IDi = Mi["ID"]
        topic = Mi["topic"]
        region = Mi.get("region", "default")

        kw1 = hashlib.sha256(IDi.encode()).hexdigest()
        kw2 = hashlib.sha256(topic.encode()).hexdigest()
        kw3 = hashlib.sha256(str(epoch).encode()).hexdigest()
        kw4 = hashlib.sha256((topic + region).encode()).hexdigest()

        ObjID = hashlib.sha256(
            (IDi + str(Mi["ts"]) + str(Mi["ctr"])).encode()
        ).hexdigest()

        index.append({
            "keywords": [kw1, kw2, kw3, kw4],
            "ObjID": ObjID
        })

    return index

def store_index(self, index, base_dir, group_k, node_j, epoch):
    import json

    path = get_storage_path(base_dir, group_k, node_j, epoch)

    with open(os.path.join(path, "index.json"), "w") as f:
        json.dump(index, f)

GLOBAL_INDEX = []

def update_global_index(index, epoch, node_j, group_k):
    for entry in index:
        GLOBAL_INDEX.append({
            "epoch": epoch,
            "node": node_j,
            "group": group_k,
            "keywords": entry["keywords"],
            "ObjID": entry["ObjID"]
        })

def search_by_topic(topic):
    kw = hashlib.sha256(topic.encode()).hexdigest()

    results = []

    for entry in GLOBAL_INDEX:
        if kw in entry["keywords"]:
            results.append(entry)

    return results

# Decryption and Verification Simulation


In [21]:

# Phase 6: Decryption and Verification at Receiver (HDFS Client)

class HDFSClient:
    def __init__(self, receiver_id, sk_hdfs, enclave_public_keys):
        self.receiver_id = receiver_id
        self.sk_hdfs = sk_hdfs
        self.enclave_public_keys = enclave_public_keys
        # Initialize KeyEncapsulation with the stored secret key
        self.kem = oqs.KeyEncapsulation(kyber_alg, self.sk_hdfs)
        
    def recover_shared_secret(self, CT_Ke):
        """Step 1: Recover SSe ← Kyber.Decaps(SK_s, CT_Ke)"""
        print(f"[HDFSClient {self.receiver_id}] Step 1: Kyber Decapsulation")
        try:
            SSe = self.kem.decap_secret(CT_Ke)
            print(f"  Recovered Shared Secret (first 16 bytes): {SSe[:16].hex()}")
            return SSe
        except Exception as e:
            print(f"  ERROR: Decapsulation failed - {e}")
            return None
    
    def derive_epoch_key(self, SSe):
        """Step 2: Derive Ke ← KDF(SSe)"""
        print(f"[HDFSClient {self.receiver_id}] Step 2: Key Derivation Function")
        # Must match enclave's derive_key which uses SHA256
        Ke_derived = hashlib.sha256(SSe).digest()
        print(f"  Derived Epoch Key (first 16 bytes): {Ke_derived[:16].hex()}")
        return Ke_derived
    
    def recover_batch_key(self, EncKe, Ke_derived):
        """Step 3: Recover batch key: Ke = EncKe ⊕ K'e"""
        print(f"[HDFSClient {self.receiver_id}] Step 3: Recover Batch Key via XOR")
        Ke_recovered = bytes(a ^ b for a, b in zip(EncKe, Ke_derived))
        print(f"  Recovered Batch Key (first 16 bytes): {Ke_recovered[:16].hex()}")
        return Ke_recovered
    
    def verify_signature(self, enclave_id, sigma_e, h_e):
        """Step 4: Verify signature: Dilithium.Verify(PK_sig, σe, he)"""
        print(f"[HDFSClient {self.receiver_id}] Step 4: Dilithium Signature Verification")
        
        if enclave_id not in self.enclave_public_keys:
            print(f"  ERROR: No public key found for enclave {enclave_id}")
            return False
        
        public_key = self.enclave_public_keys[enclave_id]
        verifier = oqs.Signature(dilithium_alg)
        
        try:
            h_e_bytes = h_e.encode() if isinstance(h_e, str) else h_e
            is_valid = verifier.verify(h_e_bytes, sigma_e, public_key)
            
            if is_valid:
                print(f"  ✓ Signature VERIFIED for enclave {enclave_id}")
                return True
            else:
                print(f"  ✗ Signature INVALID for enclave {enclave_id}")
                return False
        except Exception as e:
            print(f"  ERROR: Signature verification failed - {e}")
            return False
    
    def decrypt_batch(self, Ke, IVe, CTe, ctxe):
        """Step 5: Decrypt batch: Cagg_e ← AES-GCM.Dec(Ke, CTe, ctxe)"""
        print(f"[HDFSClient {self.receiver_id}] Step 5: AES-GCM Batch Decryption")
        
        try:
            aesgcm = AESGCM(Ke)
            Cagg_e = aesgcm.decrypt(IVe, CTe, ctxe.encode())
            print(f"  ✓ Batch decrypted successfully ({len(Cagg_e)} bytes)")
            return Cagg_e
        except Exception as e:
            print(f"  ERROR: Decryption failed - {e}")
            return None
    
    def parse_aggregated_ciphertext(self, Cagg_e):
        """Parse aggregated ciphertext back into individual messages"""
        print(f"[HDFSClient {self.receiver_id}] Parsing aggregated ciphertext")
        
        messages = []
        offset = 0
        
        while offset < len(Cagg_e):
            if offset + 4 > len(Cagg_e):
                break
            
            length = int.from_bytes(Cagg_e[offset:offset+4], 'big')
            offset += 4
            
            if offset + length > len(Cagg_e):
                break
            
            Ci = Cagg_e[offset:offset+length]
            offset += length
            messages.append(Ci)
        
        print(f"  Parsed {len(messages)} encrypted records from batch")
        return messages
    
    def process_received_batch(self, Omega_e, enclave_id):
        """Complete receiver workflow"""
        print(f"\n{'='*70}")
        print(f"[HDFS Client] Processing received batch from Epoch {Omega_e['epoch']}")
        print(f"{'='*70}\n")
        
        # Step 1: Recover shared secret via Kyber
        SSe = self.recover_shared_secret(Omega_e["CT_Ke"])
        if SSe is None:
            print("  FAILED: Could not recover shared secret")
            return None
        
        # Step 2: Derive epoch key
        Ke_derived = self.derive_epoch_key(SSe)
        
        # Step 3: Recover batch key (XOR with encrypted key from enclave)
        Ke_recovered = self.recover_batch_key(Omega_e["CT_ABE"]["ciphertext"], Ke_derived)
        
        # Step 4: Verify signature
        signature_valid = self.verify_signature(enclave_id, Omega_e["sigma_e"], Omega_e["h_e"])
        if not signature_valid:
            print("  WARNING: Signature verification failed")
            return None
        
        # Step 5: Decrypt batch
        Cagg_e = self.decrypt_batch(
            Ke_recovered,
            Omega_e["IVe"],
            Omega_e["CTe"],
            Omega_e["ctxe"]
        )
        
        if Cagg_e is None:
            print("  FAILED: Could not decrypt batch")
            return None
        
        # Parse aggregated ciphertext
        encrypted_records = self.parse_aggregated_ciphertext(Cagg_e)
        
        print(f"\n{'='*70}")
        print(f"[HDFS Client] Decryption & Verification Complete")
        print(f"{'='*70}")
        print(f"✓ Batch epoch: {Omega_e['epoch']}")
        print(f"✓ Signature verified: YES")
        print(f"✓ Records recovered: {len(encrypted_records)}")
        print(f"✓ Batch integrity: VERIFIED")
        print(f"✓ Aggregated Ciphertext: {Cagg_e}")
        print(f"✓ Encrypted Records: {encrypted_records}")
        
        return {
            "epoch": Omega_e["epoch"],
            "SSe": SSe,
            "Ke": Ke_recovered,
            "Cagg_e": Cagg_e,
            "encrypted_records": encrypted_records,
            "signature_verified": True,
            "num_records": len(encrypted_records)
        }

# Execution: HDFS Client Decryption & Verification

print("\n" + "="*70)
print("PHASE 6: RECEIVER DECRYPTION & VERIFICATION")
print("="*70 + "\n")

# Initialize HDFS Client with enclave public keys
enclave_public_keys = {}
for node in cluster.nodes:
    for enclave in node.enclaves:
        enclave_public_keys[enclave.id] = enclave.public_key

hdfs_client = HDFSClient("HDFS_Client_1", hdfs_SKs, enclave_public_keys)

# Process the received batch with timing
t_phase6_start = time.time()
received_batch = hdfs_client.process_received_batch(result, result.get("enclave_id", "Node_0-0"))
t_phase6_end = time.time()

# Store Phase 6 metrics
performance_metrics["phase_6"] = {
    "total_phase6": t_phase6_end - t_phase6_start,
    "kyber_decap": "included",
    "signature_verify": "included",
    "aes_decrypt": "included"
}

print("\n" + "="*70)
print("FINAL VERIFICATION SUMMARY")
print("="*70)
if received_batch:
    print(f"\n✓ Batch Processing Complete:")
    print(f"  • Epoch: {received_batch['epoch']}")
    print(f"  • Kyber Decapsulation: ✓ SUCCESS")
    print(f"  • Key Derivation: ✓ SUCCESS")
    print(f"  • Batch Key Recovery: ✓ SUCCESS")
    print(f"  • Signature Verification: ✓ SUCCESS")
    print(f"  • Batch Decryption: ✓ SUCCESS")
    print(f"  • Records Recovered: {received_batch['num_records']}")
    print(f"  • Overall Status: ✓ ALL STEPS PASSED\n")
else:
    print("\n✗ Batch processing FAILED\n")



PHASE 6: RECEIVER DECRYPTION & VERIFICATION


[HDFS Client] Processing received batch from Epoch 889047110

[HDFSClient HDFS_Client_1] Step 1: Kyber Decapsulation
  Recovered Shared Secret (first 16 bytes): 5d51f05f80bf0705d7f10f5d6829deee
[HDFSClient HDFS_Client_1] Step 2: Key Derivation Function
  Derived Epoch Key (first 16 bytes): cf2736d52868268a10af535d3758711d
[HDFSClient HDFS_Client_1] Step 3: Recover Batch Key via XOR
  Recovered Batch Key (first 16 bytes): b1c19ae28d29512d425df7bcbf533468
[HDFSClient HDFS_Client_1] Step 4: Dilithium Signature Verification
  ✓ Signature VERIFIED for enclave Node_0-0
[HDFSClient HDFS_Client_1] Step 5: AES-GCM Batch Decryption
  ✓ Batch decrypted successfully (140 bytes)
[HDFSClient HDFS_Client_1] Parsing aggregated ciphertext
  Parsed 6 encrypted records from batch

[HDFS Client] Decryption & Verification Complete
✓ Batch epoch: 889047110
✓ Signature verified: YES
✓ Records recovered: 6
✓ Batch integrity: VERIFIED
✓ Aggregated Ciphertext: b'\x